## Our Results

In [10]:
import os
import re
import ast
import pandas as pd
from networkx.algorithms.bipartite.basic import color

models = ['sasrec', 'caser', 'gru']
combs = [[1,1,1],[1,1,0],[1,0,1],[0,1,1],[0,1,0],[0,0,1]]
model_stop = ["main", "target"]
datasets = ["rc15_results"]
# MODIFY PATH TO div4rec dir of Paparellas
div4rec_path = "/home/marek/Kinit/my_smorl/data"
results_dir = "/home/marek/Kinit/my_smorl/Plots/Havrila_both"
os.makedirs(results_dir, exist_ok=True)


In [16]:
patterns = {
    # Matches: cumulative reward @ 5: 8585.000000
    'cumulative_reward': re.compile(r'.*cumulative reward @ (\d+): ([\d.]+)$'),

    # Matches: clicks hr ndcg @ 10 : 0.398485, 0.241570
    'clicks_hr_ndcg': re.compile(r'.*clicks hr ndcg @ (\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: purchase hr and ndcg @10 : 0.535500, 0.337177
    'purchase_hr_ndcg': re.compile(r'.*purchase hr and ndcg @(\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: total diversity reward: 48740.039062
    'total_diversity': re.compile(r'.*total diversity reward: ([\d.]+)$'),

    # Matches: total novelty reward: 19004.000000
    'total_novelty': re.compile(r'.*total novelty reward: ([\d.]+)$'),

    # Matches: coverage of top 5 predictions: 0.379410
    'coverage': re.compile(r'.*coverage of top (\d+) predictions: ([\d.]+)$'),

    # Matches: coverage on novel items of top 10 predictions: 0.385095
    'novel_coverage': re.compile(r'.*coverage on novel items of top (\d+) predictions: ([\d.]+)$'),

    # Matches: average number of repetitions in top 20: 53.379500
    'avg_repetitions': re.compile(r'.*average number of repetitions in top (\d+): \s*({.*})'),
    
    'steps': re.compile(r'.*Step: (\d+)\.+\s+Loss: ([\d.]+)$'),
    'Slosses': re.compile(r'.*Supervised loss is ([\d.]+), SMORL loss is ([\d.]+)'),
    'Mlosses': re.compile(r'.*Supervised loss is ([\d.]+), MORL loss is ([\d.]+)')
}

In [17]:
def process_results(data_path, results_path, variant, dats):
    eval_step = {'rc15_results': 5000, 'retail_rocket_results': 10000}
    
    loss = {
        'steps': [], 'loss': [], 'plain': [], 'smorl': []
    }
    metrics = {
        'steps': [],
        'hr_test_5': [], 'hr_test_10': [], 'hr_test_20': [],
        'ndcg_test_5': [], 'ndcg_test_10': [], 'ndcg_test_20': [],
        'cov_test_1': [], 'cov_test_5': [], 'cov_test_10': [], 'cov_test_20': [],
        'nov_test_1': [], 'nov_test_5': [], 'nov_test_10': [], 'nov_test_20': [],
        'rep_test_5': [], 'rep_test_10': [], 'rep_test_20': [],        
        'hr_val_5': [], 'hr_val_10': [], 'hr_val_20': [], 
        'ndcg_val_5': [], 'ndcg_val_10': [], 'ndcg_val_20': [],
        'cov_val_1': [], 'cov_val_5': [], 'cov_val_10': [], 'cov_val_20': [],        
        'nov_val_1': [], 'nov_val_5': [], 'nov_val_10': [], 'nov_val_20': [],
        'rep_val_5': [], 'rep_val_10': [], 'rep_val_20': [] 
    }
    with open(data_path, "r") as f:
        lines = f.readlines()
        my_model = "any"  
        for line in lines:
            #match = re.search(r"Step: (\d+)\.+\s+Loss: ([\d.]+)", line)
            if m := patterns['Slosses'].search(line):
                plain, smorl = float(m.group(1)), float(m.group(2))
                loss['plain'].append(plain)
                loss['smorl'].append(smorl)
            if m := patterns['Mlosses'].search(line):
                plain, smorl = float(m.group(1)), float(m.group(2))
                loss['plain'].append(plain)
                loss['smorl'].append(smorl)
            if m := patterns['steps'].match(line):
                step, _loss = int(m.group(1)), float(m.group(2))
                loss['steps'].append(step)
                loss['loss'].append(_loss)
            if "Model is" in line:
                my_model = "any"
            if "Evaluating Target Model" in line:
                my_model = "target"
            if "Evaluating Main Model" in line:
                my_model = "main"
            if my_model == variant:
                continue   
            if "TEST" in line:
                if m := patterns['clicks_hr_ndcg'].match(line):
                    k, hr, ndcg = int(m.group(1)), float(m.group(2)), float(m.group(3))
                    metrics[f'hr_test_{k}'].append(hr)
                    metrics[f'ndcg_test_{k}'].append(ndcg)
                elif m := patterns['coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'cov_test_{k}'].append(val)
                elif m := patterns['novel_coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'nov_test_{k}'].append(val)
                elif m := patterns['avg_repetitions'].match(line):
                    k, val = int(m.group(1)), m.group(2)
                    repetitions = ast.literal_eval(val)
                    value = repetitions.get(5)
                    metrics[f'rep_test_{k}'].append(value)
            else:
                if m := patterns['clicks_hr_ndcg'].match(line):
                    k, hr, ndcg = int(m.group(1)), float(m.group(2)), float(m.group(3))
                    metrics[f'hr_val_{k}'].append(hr)
                    metrics[f'ndcg_val_{k}'].append(ndcg)
                elif m := patterns['coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'cov_val_{k}'].append(val)
                elif m := patterns['novel_coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'nov_val_{k}'].append(val)
                elif m := patterns['avg_repetitions'].match(line):
                    k, val = int(m.group(1)), m.group(2)
                    repetitions = ast.literal_eval(val)
                    value = repetitions.get(5)
                    metrics[f'rep_val_{k}'].append(value)
        metrics['steps'] = [eval_step[dats] * (i + 1) for i in range(len(metrics['hr_test_5']))]
    
    return loss, metrics

for model in models:
    for comb in combs:
        for variant in model_stop:
            for dats in datasets:
                if "rc15" in dats:
                    file_path = f'{div4rec_path}/{model}_RL/{model}_smorl1_acc{comb[0]}.0_div{comb[1]}.0_nov{comb[2]}.0_weighted_q_vals.txt'
                else:
                    file_path = f'{div4rec_path}/{dats}/{model}smorl/{model}_smorl1_acc{comb[0]}.0_div{comb[1]}.0_nov{comb[2]}.0_weighted_q_vals.txt'
                results_path = f'{results_dir}/{dats}/{model}'
                os.makedirs(results_path, exist_ok=True)
                results = f"{results_path}/rl_{comb[0]}{comb[1]}{comb[2]}_{variant}"
                losses, metrics = process_results(file_path, results, variant, dats)
                df_loss = pd.DataFrame(losses)
                df_metrics = pd.DataFrame(metrics)
                df_loss.to_pickle(f'{results}_loss')
                df_loss.to_csv(f'{results}_loss.csv')
                df_metrics.to_pickle(f'{results}_metrics')
                df_metrics.to_csv(f'{results}_metrics.csv')
                
models.append('nextitnet')
for model in models:
    for dats in datasets:
        file_path = f'/home/marek/Kinit/my_smorl/data/{model}_vanilla_sk1/{model}.txt'
        results = f"{results_dir}/{dats}/{model}"
        os.makedirs(results, exist_ok=True)
        losses, metrics = process_results(file_path, results, "This is not rl - not distinguished", dats)
        losses = {k: v for k, v in losses.items() if k not in {'plain', 'smorl'}}
        df_loss = pd.DataFrame(losses)
        df_metrics = pd.DataFrame(metrics)
        df_loss.to_pickle(f'{results}/base_loss')
        df_loss.to_csv(f'{results}/base_loss.csv')
        df_metrics.to_pickle(f'{results}/base_metrics')
        df_metrics.to_csv(f'{results}/base_metrics.csv')


model = "nextitnet"
for variant in model_stop:
    for dats in datasets:
        if "rc15" in dats:
            file_path = f'{div4rec_path}/{model}_RL/{model}_smorl1_acc1.0_div1.0_nov1.0.txt'
        else:
            file_path = f'{div4rec_path}/{dats}/{model}smorl/{model}_smorl1_acc1.0_div1.0_nov1.0.txt'
        results = f"{results_dir}/{dats}/{model}/rl_111_{variant}"
        losses, metrics = process_results(file_path, results, variant, dats)
        df_loss = pd.DataFrame(losses)
        df_metrics = pd.DataFrame(metrics)
        df_loss.to_pickle(f'{results}_loss')
        df_loss.to_csv(f'{results}_loss.csv')  
        df_metrics.to_pickle(f'{results}_metrics')
        df_metrics.to_csv(f'{results}_metrics.csv')

In [426]:
points = {
    'rc15_results': {
        'gru': { 
            'base': 10000,
            '111': 10000,
            '001': 25000,
            '010': 25000,
            '011': 30000,
            '110': 25000,
            '101': 25000
        },
        'caser': {
            'base': 10000,
            '111': 75000,
            '001': 90000,
            '010': 40000,
            '011': 30000,
            '110': 110000,
            '101': 130000    
        },
        'sasrec': {
            'base': 25000,
            '111': 105000,
            '001': 115000,
            '010': 130000,
            '011': 115000,
            '110': 165000,
            '101': 115000  
        },
        'nextitnet': {
            'base': 10000,
            '111': 25000,
        }
    },
    'retail_rocket_results': {
        'gru': { 
            'base': 10000,
            '111': 30000,
            '001': 20000,
            '010': 20000,
            '011': 20000,
            '110': 30000,
            '101': 20000
        },
        'caser': {
            'base': 20000,
            '111': 50000,
            '001': 40000,
            '010': 50000,
            '011': 70000,
            '110': 40000,
            '101': 50000    
        },
        'sasrec': {
            'base': 10000,
            '111': 30000,
            '001': 30000,
            '010': 30000,
            '011': 30000,
            '110': 30000,
            '101': 30000  
        },
        'nextitnet': {
            'base': 10000,
            '111': 50000,
        }
    }
}

def get_max(dffile, testorval, alt=False):
    if alt:
        filtered = dffile[dffile['steps'] > 30000]
    else:
        filtered = dffile[dffile['steps'] > 0]
    max_row = filtered.loc[filtered['ndcg_val_10'].idxmax()]
    max_step = max_row['steps']
    max_ndcg = max_row[f'ndcg_{testorval}_10']
    hr = max_row[f'hr_{testorval}_10']
    cov = max_row[f'cov_{testorval}_10']
    nov = max_row[f'nov_{testorval}_10']
    rep = max_row[f'rep_{testorval}_10']
    values = [max_step, max_ndcg, hr, cov, nov, rep]
    return values 

def compare_values(vanilla, rl):
    winners = []
    vanilla = vanilla[1:]
    rl = rl[1:]
    for i in range(len(vanilla)-1):
        if vanilla[i] > rl[i]:
            winners.append('BM')
        else:
            winners.append('RL')
    if vanilla[-1] < rl[-1]:
        winners.append('BM')
    else:
        winners.append('RL')
    return winners
    

In [20]:
import pandas as pd
from matplotlib import pyplot as plt

def plot_helper(ax, df0, dfs, df6, PLOT, col1, col2, window=1):
    ax.set_ylabel(PLOT, color=col1)
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color=col1, linestyle=":", linewidth=2)
    for frame in dfs:
        ax.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color=col2, linewidth=1)
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color=col1, linewidth=1)
    

def plot_metrics(basepath, dataset, model, replica, testorval, window=1):
    filepath = f"{basepath}/{dataset}/{model}"
    dataline = "metrics"
    df0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        df1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        df2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        df3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        df4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        df5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfs = [df1, df2, df3, df4, df5]
    except:
        dfs = []
    df6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}")
    
    '''
    alt_base = get_max(df0, testorval, alt=True,)
    base = get_max(df0, testorval, alt=False)
    rl = get_max(df6, testorval, alt=False)
    win1 = compare_values(base, rl)
    win2 = compare_values(alt_base, rl)
    '''
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(f'Havrila : {dataset}-{model}-{replica}-{testorval}', fontsize=12, y=0.94)
     
    PLOT = f"cov_{testorval}_10"
    ax1 = axs[0, 0]
    ax1.set_title("COV 10 / NOV 10", fontsize=10, pad=10)
    ax1.set_ylim(0.1, 0.75)
    ax1.set_ylabel(PLOT, color="black")
    ax1.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label="vanilla")
    for frame in dfs:
        ax1.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=1)
    ax1.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color="black", linewidth=1, label="RL")
    
    PLOT = f"nov_{testorval}_10"
    ax2 = ax1.twinx()
    ax2.set_ylim(0.1, 0.75)
    plot_helper(ax2, df0, dfs, df6, PLOT, 'green', 'lightgreen')
       
    #~~~~~~ Plot cov + nov aligned ~~~~~~
    PLOT = f"cov_{testorval}_10"
    ax3 = axs[0, 1]
    ax3.set_title("COV 10 / NOV 10 Alignment", fontsize=10, pad=10)
    ax3.set_ylabel(PLOT, color='black')
    ax3.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2)
    for frame in dfs:
        ax3.plot(frame['steps'], frame[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)
    ax3.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='lightgray', linewidth=4)

    PLOT = f"nov_{testorval}_10"
    ax4 = ax3.twinx()
    plot_helper(ax4, df0, dfs, df6, PLOT, 'green', 'green')
    
    #~~~~~~ Plot hr + ndcg ~~~~~~   
    PLOT = f"hr_{testorval}_10"
    ax5 = axs[1, 0]
    ax5.set_title("HR 10 / NDCG 10", fontsize=10, pad=10)
    plot_helper(ax5, df0, dfs, df6, PLOT, 'black', 'lightgray')
        
    PLOT = f"ndcg_{testorval}_10"
    ax6 = ax5.twinx()
    plot_helper(ax6, df0, dfs, df6, PLOT, 'green', 'lightgreen') 
        
    # ~~~~~~ Plot repetitivness ~~~~~~
    PLOT = f"rep_{testorval}_5"
    ax7 = axs[1, 1]
    ax7.set_title("REP 5", fontsize=10, pad=10)
    ax7.yaxis.set_label_position("right")
    ax7.yaxis.tick_right()  
    plot_helper(ax7, df0, dfs, df6, PLOT, 'black', 'lightgray')
    
    # ~~~~~~ Plot losses ~~~~~~
    dataline = "loss"
    dfl0 = pd.read_pickle(f"{filepath}/base_{dataline}")
    try:
        dfl1 = pd.read_pickle(f"{filepath}/rl_001_{replica}_{dataline}")
        dfl2 = pd.read_pickle(f"{filepath}/rl_010_{replica}_{dataline}")
        dfl3 = pd.read_pickle(f"{filepath}/rl_011_{replica}_{dataline}")
        dfl4 = pd.read_pickle(f"{filepath}/rl_110_{replica}_{dataline}")
        dfl5 = pd.read_pickle(f"{filepath}/rl_101_{replica}_{dataline}")
        dfls = [dfl1, dfl2, dfl3, dfl4, dfl5]
    except:
        dfls = []
    dfl6 = pd.read_pickle(f"{filepath}/rl_111_{replica}_{dataline}") 
    
    PLOT = "loss"
    ax8 = axs[2, 0]
    ax8.set_title("Loss", fontsize=10, pad=10)
    ax8.yaxis.set_label_position("left")
    ax8.set_ylim(2, 10)
    ax8.yaxis.tick_left()
    plot_helper(ax8, dfl0, dfls, dfl6, PLOT, 'black', 'gray', 5)
          
    PLOT = "plain"
    ax9 = axs[2, 1]
    ax9.set_title("Loss components", fontsize=10, pad=10)
    ax9.set_ylim(0, 7)
    for frame in dfls:
        ax9.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgray', linewidth=1)
    ax9.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='black', linewidth=1)

    PLOT = "smorl"
    ax10 = ax9.twinx()
    ax10.set_ylabel(PLOT, color='green')
    ax10.set_ylim(0, 7)
    for frame in dfls:
        ax10.plot(frame['steps'], frame[PLOT].rolling(5).mean(), color='lightgreen', linewidth=1)
    ax10.plot(dfl6['steps'], dfl6[PLOT].rolling(5).mean(), color='green', linewidth=1)
    
    
    for i, ax in enumerate(axs.flat):
        #ax.axvline(x=points[dataset][model]['111'], color='blue', linewidth=0.5, label="evaluation_point_RL" if i == 0 else None)
        #ax.axvline(x=points[dataset][model]['base'], color='blue', linestyle=":", linewidth=0.5, label="evaluation_point_vanilla" if i == 0 else None)
        #ax.axvline(x=alt_base[0], color='red', linestyle=":", linewidth=0.5, label="alternative_point_vanilla" if i == 0 else None)
        #ax.axvline(x=max_step_RL, color='red', linewidth=0.5, label="alternative_max_RL" if i == 0 else None)  # Only label on first plot
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
        for child in ax.figure.axes:
            child.tick_params(axis='y', labelsize=8)
    
    fig.legend(
        loc='lower center',
        ncol=2,                  # number of columns in legend
        bbox_to_anchor=(0.5, -0.02),  # center below figure
        fontsize='medium'
    )
    '''
    fig.text(0.1, -0.06, 
             f"Base: NDCG: {base[1]:5.3f}, HR: {base[2]:5.3f}, COV: {base[3]:5.3f}, NOV: {base[4]:5.3f}, REP: {base[5]:5.3f}\n AltB: NDCG: {alt_base[1]:5.3f}, HR: {alt_base[2]:5.3f}, COV: {alt_base[3]:5.3f}, NOV: {alt_base[4]:5.3f}, REP: {alt_base[5]:5.3f}\n RL_1: NDCG: {rl[1]:5.3f}, HR: {rl[2]:5.3f}, COV: {rl[3]:5.3f}, NOV: {rl[4]:5.3f}, REP: {rl[5]:5.3f}",
             ha='left', fontsize=10)
    fig.text(0.9, -0.06, f"Winners: ndcg, hr, cov, nov, rep: \n Winners base: {win1}\n Winners AltB: {win2}",
             ha='right', fontsize=10)
    '''
    fig.savefig(f"{basepath}{dataset}-{model}-{replica}-{testorval}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

models = ["gru", "caser", "sasrec", "nextitnet"]
basepath = "/home/marek/Kinit/my_smorl/Plots/Havrila_both/"
for dataset in datasets:
    for model in models: 
        for replica in ['main', 'target']:
            for variant in ['test', 'val']:
                plot_metrics(basepath, dataset, model, replica, variant)
            